In [ ]:
!pip install transformers
!pip install pytorch-crf -i https://pypi.tuna.tsinghua.edu.cn/simple/
!pip install datasets

In [ ]:
!pip install seqeval -q

In [ ]:
from transformers import BertTokenizer, BertModel,BertTokenizerFast,AutoModelForTokenClassification
import torch
import torch.nn as nn
import torchcrf

In [ ]:
#tokenizer = BertTokenizer.from_pretrained("hfl/chinese-roberta-wwm-ext-large")
tokenizer = BertTokenizerFast.from_pretrained("ckiplab/bert-base-chinese")
#tokenizer 可換成其他模型

In [ ]:
device = "cuda" if torch.cuda.is_available()  else "cpu"
print(device)

In [ ]:
#pre_model = BertModel.from_pretrained("hfl/chinese-roberta-wwm-ext-large")
pre_model = BertModel.from_pretrained("ckiplab/bert-base-chinese")
#計算模型 可替換

In [ ]:
import pandas as pd

In [ ]:
train_data = pd.read_json("train.json", lines=True)
valid_data = pd.read_json("test.json", lines=True)
test_data = pd.read_json("conference_test.json")

In [ ]:
train_data.head()

# **training prepocessing**

In [ ]:
def collate_fn(data):
    sents = [i for i in data["sentence"]]
    labels = data["character_label"]  #entity標籤
    
    inputs = tokenizer(sents, return_tensors="pt", padding=True, truncation=True, max_length=150) #-------------------------------->可修改
    input_ids = inputs["input_ids"] #token
    attention_mask = inputs["attention_mask"] #mask

    lens = input_ids.shape[1] #等同於max_length

    for i in range(len(labels)):#資料筆數
      #對每個標籤序列（labels），添加特殊標籤（21）以填充到最大長度（lens）的次數。這是為了對應輸入的token序列。
      labels[i] += [21] * lens 
      labels[i] = labels[i][:lens]

    return input_ids, attention_mask, labels
    
inputs = collate_fn(train_data)

In [ ]:
inputt = inputs[0]
att = inputs[1]
label = inputs[2]

In [ ]:
def get_tag2index():
    return {"O": 0,
        "B-BODY":1,
        "I-BODY":2,
        "B-SYMP":3,
        "I-SYMP":4,     
        "B-INST":5,
        "I-INST":6,
        "B-EXAM":7,
        "I-EXAM":8,
        "B-CHEM":9,
        "I-CHEM":10,
        "B-DISE":11,
        "I-DISE":12,
        "B-DRUG":13,
        "I-DRUG":14,
        "B-SUPP":15,
        "I-SUPP":16,
        "B-TREAT":17,
        "I-TREAT":18,
        "B-TIME":19,
        "I-TIME":20,  #命名增加須手動調整
        21:21}     #21 為填充
def gget_tag2index():
    return {"O": 1,
        "B-BODY":1,
        "I-BODY":1,
        "B-SYMP":1,
        "I-SYMP":1,
        "B-INST":1,
        "I-INST":1,
        "B-EXAM":1,
        "I-EXAM":1,
        "B-CHEM":1,
        "I-CHEM":1,
        "B-DISE":1,
        "I-DISE":1,
        "B-DRUG":1,
        "I-DRUG":1,
        "B-SUPP":1,
        "I-SUPP":1,
        "B-TREAT":1,
        "I-TREAT":1,
        "B-TIME":1,
        "I-TIME":1,   #label 遮罩crf 句子截斷判斷需用 1: 留下 0: 去除
        21:0}

In [ ]:
tag2i = get_tag2index()
tag2i_1 = gget_tag2index()


lab = []
lab_1 = []

for i in range(inputs[0].shape[0]):
          #資料筆數
  tag_index = [tag2i.get(l,0) for l in label[i]] #將資料內label轉為對應數字
  lab.append(tag_index)

  tag_index_1 = [tag2i_1.get(l,0) for l in label[i]] #將資料內label轉為crf mask對應數字
  lab_1.append(tag_index_1)  

llab = torch.LongTensor(lab) #轉為tensor格式
llab_1 = torch.LongTensor(lab_1) 
  


In [ ]:
from torch.utils.data import Dataset, DataLoader, RandomSampler

In [ ]:
class MiDataset(Dataset):
  def __init__(self, texts, labels, masks, masks_1):
    self.dataset = texts
    self.labels = labels
    self.masks = masks
    self.masks_1 = masks_1

    self.nums = len(self.dataset)
  
  def __getitem__(self, index):
    data = {"texts" : self.dataset[index],
         "labels" : self.labels[index],
         "masks" : self.masks[index],
         "masks_1" : self.masks_1[index]}
    return data

  def __len__(self):
    return self.nums                

In [ ]:
texts = torch.LongTensor(inputt)
labels = llab
masks = torch.LongTensor(att).type(torch.ByteTensor)
masks_1 = llab_1.type(torch.ByteTensor)

print("---------texts.shape句子----------")
print(texts.shape)
print("---------labels.shape對應標號----------")
print(labels.shape)
print("---------masks.shape遮罩(01010)----------")
print(masks.shape)
print("---------masks_1.shape遮罩(01010) for crf----------")
print(masks_1.shape)

In [ ]:
train_data = MiDataset(texts, labels, masks, masks_1)
train_data

In [ ]:
train_loader = DataLoader(dataset = train_data,
              batch_size = 16, #--------------------------->批量大小(可以改)要和下方一起改
              num_workers = 0,
              shuffle = False,
              drop_last = False)
train_loader

# **test prepocessing**
基本動作同訓練資料

In [ ]:
test_data["sentence"] = test_data["character"].apply(lambda x : "".join(x))
test_data.head()

In [ ]:
test_inputs = collate_fn(test_data)

In [ ]:
test_inputt = test_inputs[0]
test_att = test_inputs[1]
test_label = pd.array(test_inputs[2])

In [ ]:
test_lab = []
test_lab_1 = []

test_tag2i = get_tag2index()    #獲得tag_to_index辭典
test_tag2i_1 = gget_tag2index()

for i in range(test_inputs[0].shape[0]):
  test_tag_index = [test_tag2i.get(l,0) for l in test_label[i]]
  test_lab.append(test_tag_index)
test_llab = torch.LongTensor(test_lab)

for i in range(test_inputs[0].shape[0]):
  test_tag_index_1 = [test_tag2i_1.get(l,0) for l in test_label[i]]
  test_lab_1.append(test_tag_index_1)
test_llab_1 = torch.LongTensor(test_lab_1)


In [ ]:
texts1 = torch.LongTensor(test_inputt)
test_labels = test_llab
test_masks = torch.LongTensor(test_att).type(torch.ByteTensor)
test_masks_1 = test_llab_1.type(torch.ByteTensor)

In [ ]:
#將數據集用MiDataset類包裝
test_data = MiDataset(texts1,test_labels,test_masks,test_masks_1)
test_data

#train_sampler = RandomSampler(train_data)    #將訓練集打亂
test_loader = DataLoader(dataset=test_data,   #按batch_size加載訓練集
                                batch_size=16, 
                                #sampler=train_sampler,
                                num_workers=0,
                                shuffle=False,
                                drop_last=False)

# **valid prepocessing**

In [ ]:
valid_data["sentence"] = valid_data["character"].apply(lambda x : "".join(x))
valid_data.head()

In [ ]:
valid_inputs = collate_fn(valid_data)
valid_inputt = valid_inputs[0]
valid_att = valid_inputs[1]
valid_label = pd.array(valid_inputs[2])

In [ ]:
valid_lab = []
valid_lab_1 = []

valid_tag2i = get_tag2index()    #獲得tag_to_index辭典
valid_tag2i_1 = gget_tag2index()

for i in range(valid_inputs[0].shape[0]):
  valid_tag_index = [valid_tag2i.get(l,0) for l in valid_label[i]]
  valid_lab.append(valid_tag_index)
valid_llab = torch.LongTensor(valid_lab)

for i in range(valid_inputs[0].shape[0]):
  valid_tag_index_1 = [valid_tag2i_1.get(l,0) for l in valid_label[i]]
  valid_lab_1.append(valid_tag_index_1)
valid_llab_1 = torch.LongTensor(valid_lab_1)


In [ ]:
texts1 = torch.LongTensor(valid_inputt)
valid_labels = valid_llab
valid_masks = torch.LongTensor(valid_att).type(torch.ByteTensor)
valid_masks_1 = valid_llab_1.type(torch.ByteTensor)

In [ ]:
#將數據集用MiDataset類包裝
valid_data = MiDataset(texts1,valid_labels,valid_masks,valid_masks_1)
valid_data

#train_sampler = RandomSampler(train_data)    #將訓練集打亂
valid_loader = DataLoader(dataset=valid_data,   #按batch_size加載訓練集
                                batch_size=16, 
                                #sampler=train_sampler,
                                num_workers=0,
                                shuffle=False,
                                drop_last=False)

# **model training**

In [ ]:
from sklearn.metrics import f1_score, accuracy_score, recall_score, precision_score

#batch_masks:tensor數據，結構為(batch_size,MAX_LEN)
#batch_labels: tensor數據，結構為(batch_size,MAX_LEN)
#batch_prediction:list數據，結構為(batch_size,)   #每個數據長度不一（在model參數mask存在的情況下）
def batch_labels_process(batch_masks,batch_labels):
    all_labels = []
    batch_size = batch_masks.shape[0]   #防止最後一batch的數據不够batch_size
    for index in range(batch_size):   #22
        #把没有mask掉的原始tag都集合到一起
        length = sum(batch_masks[index].cpu().numpy()==1)
        _label = batch_labels[index].numpy().tolist()[:length]
        all_labels = all_labels+_label  
        #把没有mask掉的預測tag都集合到一起
        #_predict = y_pred[index][:length]
    return all_labels

In [ ]:
def valid():
  y_labels = []
  y_pred_all = []
  for step, batch_data in enumerate(valid_loader):
      with torch.no_grad(): 
        valid_sentence = batch_data['texts'].to(device)
        valid_mask1 = batch_data['masks'].to(device)
        validmscrf = batch_data['masks_1'].to(device)
        batch_labels = batch_data['labels']
        batch_pred = model(valid_sentence, tags=None , mask=valid_mask1, mask_1=validmscrf)
        
        
        for i in batch_pred:  
          y_pred_all.append(i)
        

        batch_labels = batch_labels_process(batch_masks=validmscrf, batch_labels=batch_labels)
        
        for i in batch_labels:
          y_labels.append(i)
  y_preds = []
  for i in y_pred_all:
    for j in i:
      y_preds.append(j)

  accuracy = accuracy_score(y_preds,y_labels)
  precision = precision_score(y_preds,y_labels,average='weighted')
  recall = recall_score(y_preds,y_labels,average='weighted')
  f1 = f1_score(y_preds,y_labels,average='weighted')

  print("Valid--------------------------------------------------------------------------------------------------------------------------------------------------------------------")
  print("accuracy-score:"+str(accuracy))
  print("precision-score:"+str(precision))
  print("recall-score:"+str(recall))
  print("f1-score:"+str(f1))
  print("End--------------------------------------------------------------------------------------------------------------------------------------------------------------------")


In [ ]:
#hidden_dim = 1024
hidden_dim = 768  #bert模型維度
target_size = 22   #命名目標種類
batch_size = 16   #--------------------------->批量大小(可以改)要和上方一起改

In [ ]:
from torchcrf import CRF

In [ ]:
for param in pre_model.parameters():   #不用更新引用模型梯度
    param.requires_grad_(False)

class BiLSTM_CRF(nn.Module):
    def __init__(self, target_size, model, hidden_dim):
        super(BiLSTM_CRF, self).__init__()
        self.model = model
        self.hidden_dim = hidden_dim
        self.target_size = target_size
        self.lstm = nn.LSTM(
            input_size=hidden_dim,
            hidden_size=hidden_dim//2, 
            batch_first=True,                         
            bidirectional=True,               
            num_layers=2
        )
        self.hidden2tag2 = nn.Linear(hidden_dim, self.target_size)
        self.crf = CRF(self.target_size)
        
    def forward(self, sentence, tags=None, mask=None, mask_1=None):
        with torch.no_grad():                   
          embeds1 = self.model(sentence,mask).last_hidden_state  
        lstm_out, self.hidden = self.lstm(embeds1) 
        lstm_featss = self.hidden2tag2(lstm_out) 
        lstm_feats =lstm_featss.permute(1,0,2)

        #4. 全連接層到CRF層
        if tags is not None: #訓練用
            if mask is not None:
                loss = (-1)*self.crf.forward(emissions=lstm_feats,mask=mask_1.permute(1,0),tags=tags.permute(1,0),reduction='mean')   #outputs=(batch_size,)  輸出log形式的likelihood
            else:
                loss = (-1)*self.crf.forward(emissions=lstm_feats,tags=tags.permute(1,0),reduction='mean')
            return loss
        else:   #測試用
            if mask is not None:
              #prediction = lstm_feats.softmax(dim=2)
               prediction = self.crf.decode(emissions=lstm_feats,mask=mask_1.permute(1,0))   #mask=attention_masks.byte()  .permute(1,0) ,mask=tags
            else:
               prediction = self.crf.decode(emissions=lstm_feats)
            return prediction

In [ ]:
import torch.optim as optim

#創建模型和優化器
model = BiLSTM_CRF(target_size, pre_model, hidden_dim)
optimizer = optim.SGD(model.parameters(), lr=0.012, weight_decay=1e-5)
lr_period, lr_decay = 2, 0.9
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, lr_period, lr_decay)
model.to(device)

In [ ]:
import math
samples_cnt = texts.shape[0]
batch_cnt = math.ceil(samples_cnt/batch_size)  #整除 向上取整
loss_list = []
model.train()
for epoch in range(1):
    for step, batch_data in enumerate(train_loader):
        # 1. 清空梯度
        bt = batch_data['texts'].to(device)
        bl = batch_data['labels'].to(device)
        bm = batch_data['masks'].to(device)
        bm_1 = batch_data['masks_1'].to(device)
        
        # 2. 運行模型
        loss = model(sentence=bt, tags=bl, mask=bm, mask_1=bm_1) 
        if step%100 ==0:
            print('Epoch=%d  step=%d/%d  loss=%.5f' % (epoch,step,batch_cnt,loss))
        
        # 3. 計算loss值，梯度并更新權重參數                                 
        loss.backward()    #retain_graph=True)  #反向傳播，計算當前梯度
        optimizer.step()  #根據梯度更新網路參數
        optimizer.zero_grad()
    valid()
    scheduler.step()
    loss_list.append(loss)

In [ ]:
'''
import os

os.makedirs(os.path.join('/content/gdrive/MyDrive/Models/'), exist_ok=True) 
model_path = '/content/gdrive/MyDrive/Models/'
#保存訓練结果
current_model_path = model_path+"_model.pkl"
torch.save(model,current_model_path)
print("保存模型文件："+current_model_path)
'''

# **test**

In [ ]:
import numpy as np

model.to(device)


y_labels = []
y_pred_all = []
for step, batch_data in enumerate(test_loader):
    with torch.no_grad(): 
      test_sentence = batch_data['texts'].to(device)
      test_mask1 = batch_data['masks'].to(device)
      testmscrf = batch_data['masks_1'].to(device)
      batch_labels = batch_data['labels']
      batch_pred = model(test_sentence, tags=None , mask=test_mask1, mask_1=testmscrf)
      
      
      for i in batch_pred:  
        y_pred_all.append(i)
      

      batch_labels = batch_labels_process(batch_masks=testmscrf, batch_labels=batch_labels)
      for i in batch_labels:
        y_labels.append(i)


In [ ]:
y_preds = []
for i in y_pred_all:
  for j in i:
    y_preds.append(j)

accuracy = accuracy_score(y_preds,y_labels)
precision = precision_score(y_preds,y_labels,average='macro')
recall = recall_score(y_preds,y_labels,average='macro')
f1 = f1_score(y_preds,y_labels,average='macro')

print("accuracy-score:"+str(accuracy))
print("precision-score:"+str(precision))
print("recall-score:"+str(recall))
print("f1-score:"+str(f1))

In [ ]:
def get_tag():
    return {0:"O",
        1:"B-BODY",
        2:"I-BODY",
        3:"B-SYMP",
        4:"I-SYMP",
        5:"B-INST",
        6:"I-INST",
        7:"B-EXAM",
        8:"I-EXAM",
        9:"B-CHEM",
        10:"I-CHEM",
        11:"B-DISE",
        12:"I-DISE",
        13:"B-DRUG",
        14:"I-DRUG",
        15:"B-SUPP",
        16:"I-SUPP",
        17:"B-TREAT",
        18:"I-TREAT",
        19:"B-TIME",
        20:"I-TIME"}
get_tag = get_tag()
pred = []
for i in y_preds:
  pred.append(get_tag.get(i))

label = []
for i in y_labels:
  label.append(get_tag.get(i))


In [ ]:
pred[:20]

In [ ]:
label[:20]